In [2]:
import os
os.environ['GROQ_API_KEY']="your api key"

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain_huggingface langchain_groq faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_3755/2406505708.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Step 1a - Indexing(Document Ingestion)

In [5]:
video_id = "DF3XjEhJ40Y"

try:
    api = YouTubeTranscriptApi()

    transcript_list = api.fetch(video_id, languages=["fr"])

    transcript = " ".join(
        snippet.text for snippet in transcript_list
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

L’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air est lourd  Le regard absent  Il est seul et lui parle souvent  Il, il n'est pas fou  Il l'aime c'est tout  Il la voit partout  Il l'attend debout  Debout une rose à la main  Non, non plus rien ne le retient  Dans sa love story  Dans sa love story  Dans sa love story  Sa love story  Prends ma main  Promets-moi que tout ira bien Serre-moi fort  Près de toi, je rêve encore  Oui, oui, je veux rester  Mais je ne sais plus aimer  J'ai été trop bête  Je t'en prie, arrête  Arrête, comme je regrette  Non, je ne voulais pas tout ça  Je serai riche  Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love stor

In [6]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='L’âme en peine ', start=59.44, duration=2.76), FetchedTranscriptSnippet(text='Il vit, mais parle à peine ', start=62.2, duration=3.8), FetchedTranscriptSnippet(text='Il attend ', start=67.0, duration=2.6), FetchedTranscriptSnippet(text="devant cette photo d'antan ", start=69.6, duration=3.96), FetchedTranscriptSnippet(text="Il, il n'est pas fou ", start=73.56, duration=2.44), FetchedTranscriptSnippet(text="Il y croit c'est tout ", start=76.0, duration=1.88), FetchedTranscriptSnippet(text='Il la voit partout ', start=77.88, duration=2.12), FetchedTranscriptSnippet(text="Il l'attend debout ", start=80.0, duration=2.0), FetchedTranscriptSnippet(text='Une rose à la main ', start=82.0, duration=1.4), FetchedTranscriptSnippet(text="À part elle il n'attend rien ", start=83.4, duration=3.6), FetchedTranscriptSnippet(text='Rien autour  ', start=89.36, duration=2.64), FetchedTranscriptSnippet(text="n'a de sens ", start=92.0, duration=1.0

# step 1b: Indexing(Text Chunking)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [8]:
len(chunks)

2

In [10]:
chunks[0]

Document(metadata={}, page_content="L’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air est lourd  Le regard absent  Il est seul et lui parle souvent  Il, il n'est pas fou  Il l'aime c'est tout  Il la voit partout  Il l'attend debout  Debout une rose à la main  Non, non plus rien ne le retient  Dans sa love story  Dans sa love story  Dans sa love story  Sa love story  Prends ma main  Promets-moi que tout ira bien Serre-moi fort  Près de toi, je rêve encore  Oui, oui, je veux rester  Mais je ne sais plus aimer  J'ai été trop bête  Je t'en prie, arrête  Arrête, comme je regrette  Non, je ne voulais pas tout ça  Je serai riche  Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Da

In [12]:
chunks[1]

Document(metadata={}, page_content="Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story")

#step1 c & step1D (creatingand storing the embeddings into a vector store)

In [13]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
vector_store.index_to_docstore_id

{0: '14824f5b-3ee1-47d3-9681-c924a7bda253',
 1: 'e486740c-6643-4efc-b62b-f6af54eb3995'}

In [15]:
# if yu wanna see a perticular vector
vector_store.get_by_ids(["e486740c-6643-4efc-b62b-f6af54eb3995"])

[Document(id='e486740c-6643-4efc-b62b-f6af54eb3995', metadata={}, page_content="Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story")]

 # Step 2 - Retrival

In [17]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [19]:
retriever.invoke("what is love story")

[Document(id='e486740c-6643-4efc-b62b-f6af54eb3995', metadata={}, page_content="Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story"),
 Document(id='14824f5b-3ee1-47d3-9681-c924a7bda253', metadata={}, page_content="L’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air est lourd  Le regard a

# Step 3 - Augumentation

In [20]:
LLM_model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.7)

In [21]:
# creating the prompt template
prompt = PromptTemplate(
    template = """
    you are a french to english song expert
    Answer only from the provided transcript
    if the context is insufficient, just say you dont know

    {context}
    Question: {question}
    """,
    input_variables={"context", "question"}
)

In [22]:
question = "is this song is in english or latin in this video, if yes what waas this song about"
retrieved_docs = retriever.invoke(question)

In [24]:
retrieved_docs

[Document(id='e486740c-6643-4efc-b62b-f6af54eb3995', metadata={}, page_content="Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story"),
 Document(id='14824f5b-3ee1-47d3-9681-c924a7bda253', metadata={}, page_content="L’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air est lourd  Le regard a

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [27]:
context_text

"Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story\n\nL’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air est lourd  Le regard absent  Il est seul et lui parle souvent  Il, il n'est pas fou  Il l'aime c'est tout  Il la voit partout  Il l'attend debout  Debout une rose à la main  Non, no

In [26]:
final_report = prompt.invoke({"context": context_text, "question": question})

In [28]:
final_report

StringPromptValue(text="\n    you are a french to english song expert\n    Answer only from the provided transcript\n    if the context is insufficient, just say you dont know\n\n    Et je t'offrirai tout mon or  Et si tu t'en fiches  Je t'attendrai sur le port  Et si tu m'ignores  Je t'offrirai mon dernier  souffle de vie  Dans ma love story  Dans ma love story   Dans ma love story  Ma love story  Une bougie  Peut illuminer la nuit  Un sourire  Peut bâtir tout un empire  Et il y a toi  Et il y a moi  Et personne n'y croit  Mais l'amour fait d'un fou un roi  Et si tu m'ignores  J'me battrais encore et encore  C'est ta love story  C'est ta love story  C'est l'histoire d'une vie   Love story  Love story Love story Love story Love story\n\nL’âme en peine  Il vit, mais parle à peine  Il attend  devant cette photo d'antan  Il, il n'est pas fou  Il y croit c'est tout  Il la voit partout  Il l'attend debout  Une rose à la main  À part elle il n'attend rien  Rien autour   n'a de sens  Et l'air

# Step 4 - Generation

In [29]:
answer = LLM_model.invoke(final_report)
print(answer.content)

The song is not in English or Latin, it's in French. 

As for what the song is about, based on the provided transcript, it appears to be a love story about a person who is deeply in love and willing to offer everything they have, including their life, to the one they love. The song seems to describe a romantic and possibly unrequited love, with the speaker expressing their devotion and willingness to wait and fight for the other person. The lyrics mention a "love story" repeatedly, suggesting that the song is a narrative about a romantic relationship.


# Now we are invoking at 3 different points which is considered not a good practice so we will build a chain for this

# Building Chain

In [30]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [31]:
# we will give the context joining to a function here
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [34]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [35]:
parser = StrOutputParser()

In [36]:
main_chain = parallel_chain | prompt | LLM_model | parser

In [37]:
main_chain.invoke('Can you summarize the song in english')

'The song appears to be a romantic ballad about a person\'s all-consuming love for someone. The lyrics convey a sense of devotion, longing, and desperation. \n\nThe song\'s narrator is willing to offer their wealth ("I\'ll offer you all my gold") and even their life ("I\'ll offer you my last breath of life") for the person they love. They\'re willing to wait for this person, even if they\'re ignored or rejected ("I\'ll wait for you on the dock", "If you ignore me, I\'ll fight again and again").\n\nThe song also touches on the theme of being oblivious to reality due to love, with the narrator stating that "love makes a fool a king". \n\nThe repetition of "love story" and "ma love story" (my love story) throughout the song suggests that the narrator is deeply invested in this romantic narrative, and is willing to do whatever it takes to make it work.\n\nThe second part of the transcript appears to describe a person who is heartbroken and still holding on to memories of a past love, and i